<a href="https://colab.research.google.com/github/shrivash-raha/gemma-3-1b-lora-tuning/blob/main/gemma-3-1b-fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes torchao>=0.16.0

In [2]:
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer
)

from peft import (
    LoraConfig,
    get_peft_model,
    PeftModel
)

from datasets import load_dataset

In [87]:
model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

# Model Inspection

In [20]:
total_params = sum(
    p.numel() for p in model.parameters()
)

print(f"Total Parameters: {total_params:,}")

Total Parameters: 496,195,456


In [21]:
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print(f"Trainable Parameters: {trainable_params:,}")

Trainable Parameters: 2,162,688


In [22]:
for name, module in model.named_modules():
    if "proj" in name:
        print(name)

base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.base_layer
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default
base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_embedd

In [23]:
for name, module in model.named_modules():
    if "proj" in name:
        print(f"Module: {name}")
        for param_name, param in module.named_parameters():
            print(f"  Parameter: {param_name}, Shape: {param.shape}")

Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj
  Parameter: base_layer.weight, Shape: torch.Size([896, 896])
  Parameter: base_layer.bias, Shape: torch.Size([896])
  Parameter: lora_A.default.weight, Shape: torch.Size([16, 896])
  Parameter: lora_B.default.weight, Shape: torch.Size([896, 16])
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.base_layer
  Parameter: weight, Shape: torch.Size([896, 896])
  Parameter: bias, Shape: torch.Size([896])
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_dropout.default
Module: base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A
  Parameter: default.weight, Shape: torch.Size([16, 896])
Module: 

# LoRA Config

In [88]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [6]:
!pip install --upgrade torchao>=0.16.0

In [37]:
import peft
print(peft.__version__)

0.19.1


In [89]:
model = get_peft_model(
    model,
    lora_config
)

# LoRA Inspection

In [44]:
model.print_trainable_parameters()

trainable params: 2,162,688 || all params: 496,195,456 || trainable%: 0.4359


In [45]:
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.shape)

base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight torch.Size([896, 16])
base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight torch.Size([16, 896])
base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight torch.Size([128, 16])
base_model.model

In [46]:
for name, module in model.named_modules():
    if "q_proj" in name:
        print(name)
        print(type(module))
        break

base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.base_model.model.model.layers.0.self_attn.q_proj
<class 'peft.tuners.lora.layer.Linear'>


In [106]:
layer = model.base_model.model.model.layers[0]

print(layer.self_attn.q_proj.lora_A["default"].weight.shape)
print(layer.self_attn.q_proj.lora_B["default"].weight.shape)

torch.Size([16, 896])
torch.Size([896, 16])


In [107]:
A = layer.self_attn.q_proj.lora_A["default"].weight
B = layer.self_attn.q_proj.lora_B["default"].weight

print(A.shape)
print(B.shape)

delta_w = B @ A

print(delta_w.shape)

torch.Size([16, 896])
torch.Size([896, 16])
torch.Size([896, 896])


# Pre Training Inference

In [112]:
prompt = "Explain what LoRA is."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Explain what LoRA is. LoRA stands for Long Range Awareness, and it refers to the ability of an AI system to recognize and identify objects or people from a long distance away. It works by using machine learning algorithms that can detect and track objects or people in images or videos over large distances.
LoRA has several applications, including:
1. Augmented reality: In augmented reality, LoRA can be used to create 3D models of objects or people in real-world environments, allowing users to explore them from a safe


# Dataset

In [119]:
dataset = load_dataset(
    "yahma/alpaca-cleaned"
)

dataset = dataset["train"].select(
    range(1000)
)

In [141]:
def format_example(example):

    text = f"""
### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}
"""

    return {"text": text}

In [142]:
dataset = dataset.map(
    format_example
)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [146]:
print(dataset[0]['text'])


### Instruction:
Give three tips for staying healthy.

### Input:


### Response:
1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.

2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.

3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.



In [140]:
for val in [kv for kv in dataset if kv['input'] != '']:
  print('input: ', val['input'])
  print('instruction: ', val['instruction'])
  print('output: ', val['output'])
  print("\n\n================================================\n\n")

input:  4/16
instruction:  Explain why the following fraction is equivalent to 1/4
output:  The fraction 4/16 is equivalent to 1/4 because both fractions represent the same value. A fraction can be simplified by dividing both the numerator and the denominator by a common factor. In this case, 4 is a common factor of both the numerator and the denominator of 4/16. When we divide both by 4, we get 4/4 = 1 and 16/4 = 4, so the simplified fraction is 1/4. Alternatively, we can think of this in terms of multiplication. For example, if we multiply the numerator and denominator of the fraction 1/4 by 4, we get (1x4)/(4x4), or 4/16. Since both fractions can be derived from the other through multiplication or division by the same number, they represent the same value and are equivalent.




input:  He finnished his meal and left the resturant
instruction:  Evaluate this sentence for spelling and grammar mistakes
output:  There are two spelling errors in the sentence. The corrected sentence shou

In [147]:
from trl import SFTTrainer
from transformers import TrainingArguments

In [148]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=1,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=4,

    logging_steps=10,

    save_steps=100,

    learning_rate=2e-4,

    fp16=True
)

In [149]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args
)

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:964: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  args = SFTConfig(**dict_args)


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [150]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,1.532438
20,1.358397
30,1.454923
40,1.456585
50,1.516089
60,1.316700
70,1.331338
80,1.323638
90,1.322040
100,1.349922


TrainOutput(global_step=125, training_loss=1.394398292541504, metrics={'train_runtime': 112.2389, 'train_samples_per_second': 8.91, 'train_steps_per_second': 1.114, 'total_flos': 497363504985600.0, 'train_loss': 1.394398292541504, 'entropy': 1.3883132487535477, 'mean_token_accuracy': 0.6553211495280266, 'num_tokens': 162088.0, 'epoch': 1.0})

In [153]:
model.save_pretrained(
    "qwen_lora_adapter"
)

tokenizer.save_pretrained(
    "qwen_lora_adapter"
)

('qwen_lora_adapter/tokenizer_config.json',
 'qwen_lora_adapter/chat_template.jinja',
 'qwen_lora_adapter/tokenizer.json')

In [155]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [156]:
model = PeftModel.from_pretrained(
    base_model,
    "qwen_lora_adapter"
)

In [ ]:
prompt = "Explain what LoRA is."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100000
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)